In [13]:
! pip install -q transformers datasets accelerate evaluate scikit-learn huggingface_hub

In [14]:
import torch
import torch.nn as nn
from transformers import AutoModel, AutoTokenizer
from transformers import AutoModelForSequenceClassification

In [15]:
print(torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

False No GPU


In [16]:
model_checkpoint = "distilbert-base-uncased"
base_model = AutoModel.from_pretrained(model_checkpoint)
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [17]:
print(base_model)

DistilBertModel(
  (embeddings): Embeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (transformer): Transformer(
    (layer): ModuleList(
      (0-5): 6 x TransformerBlock(
        (attention): DistilBertSelfAttention(
          (q_lin): Linear(in_features=768, out_features=768, bias=True)
          (k_lin): Linear(in_features=768, out_features=768, bias=True)
          (v_lin): Linear(in_features=768, out_features=768, bias=True)
          (out_lin): Linear(in_features=768, out_features=768, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
        (ffn): FFN(
          (dropout): Dropout(p=0.1, inplace=False)
          (lin1): Linear(in_features=768, out_features=3072, bias=True)
          (lin2): L

In [18]:
id2label = {0: "negative", 1:"neutral", 2:"positive"}
label2id = {"negative":0, "neutral":1, "positive":2}

classification_model = AutoModelForSequenceClassification.from_pretrained(
    model_checkpoint,
     num_labels=3,
    id2label=id2label,
    label2id=label2id
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [19]:
print(classification_model)

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [20]:
print(classification_model.config)

DistilBertConfig {
  "activation": "gelu",
  "architectures": [
    "DistilBertForMaskedLM"
  ],
  "attention_dropout": 0.1,
  "bos_token_id": null,
  "dim": 768,
  "dropout": 0.1,
  "dtype": "float32",
  "eos_token_id": null,
  "hidden_dim": 3072,
  "id2label": {
    "0": "negative",
    "1": "neutral",
    "2": "positive"
  },
  "initializer_range": 0.02,
  "label2id": {
    "negative": 0,
    "neutral": 1,
    "positive": 2
  },
  "max_position_embeddings": 512,
  "model_type": "distilbert",
  "n_heads": 12,
  "n_layers": 6,
  "pad_token_id": 0,
  "qa_dropout": 0.1,
  "seq_classif_dropout": 0.2,
  "sinusoidal_pos_embds": false,
  "tie_weights_": true,
  "tie_word_embeddings": true,
  "transformers_version": "5.13.1",
  "vocab_size": 30522
}



In [21]:
# Full list of every parameter name + its shape, in the base encoder (no head)
for name, params in base_model.state_dict().items():
  print(f"{name}  ------>       {tuple(params.shape)}")

embeddings.word_embeddings.weight  ------>       (30522, 768)
embeddings.position_embeddings.weight  ------>       (512, 768)
embeddings.LayerNorm.weight  ------>       (768,)
embeddings.LayerNorm.bias  ------>       (768,)
transformer.layer.0.attention.q_lin.weight  ------>       (768, 768)
transformer.layer.0.attention.q_lin.bias  ------>       (768,)
transformer.layer.0.attention.k_lin.weight  ------>       (768, 768)
transformer.layer.0.attention.k_lin.bias  ------>       (768,)
transformer.layer.0.attention.v_lin.weight  ------>       (768, 768)
transformer.layer.0.attention.v_lin.bias  ------>       (768,)
transformer.layer.0.attention.out_lin.weight  ------>       (768, 768)
transformer.layer.0.attention.out_lin.bias  ------>       (768,)
transformer.layer.0.sa_layer_norm.weight  ------>       (768,)
transformer.layer.0.sa_layer_norm.bias  ------>       (768,)
transformer.layer.0.ffn.lin1.weight  ------>       (3072, 768)
transformer.layer.0.ffn.lin1.bias  ------>       (3072,)


In [22]:
all_keys = list(base_model.state_dict().keys())
print(f"Total number of parameter tensors: {len(all_keys)}")

# Should be: 4 (embeddings) + 6 layers x 16 params/layer = 4 + 96 = 100

Total number of parameter tensors: 100


In [63]:
import torch
import torch.nn as nn
import math

class DistilBertEmbeddings(nn.Module):
    """
    the embedding layer of DistilBERT.

    Converts input_ids into dense vectors by summing:
      1. Token embeddings (word identity)
      2. Position embeddings (position in the sequence)
    followed by LayerNorm and Dropout.
    """

    def __init__(self, vocab_size=30522, hidden_size=768, max_position_embeddings=512, dropout_prob=0.1):
        super().__init__()

        self.word_embeddings = nn.Embedding(vocab_size, hidden_size)
        self.position_embeddings = nn.Embedding(max_position_embeddings, hidden_size)
        self.LayerNorm = nn.LayerNorm(hidden_size, eps=1e-12)
        self.dropout = nn.Dropout(dropout_prob)

    def forward(self, input_ids):
        batch_size, seq_len = input_ids.shape

        token_embeds = self.word_embeddings(input_ids)

        position_ids = torch.arange(seq_len, dtype=torch.long, device=input_ids.device)
        position_ids = position_ids.unsqueeze(0).expand(batch_size, seq_len)
        position_embeds = self.position_embeddings(position_ids)

        embeddings = token_embeds + position_embeds
        embeddings = self.LayerNorm(embeddings)
        embeddings = self.dropout(embeddings)

        return embeddings

In [66]:
input_ids = tokenizer.encode(["I like this","I hate this"], max_length=128, padding="max_length", return_tensors='pt')
embedding_object = DistilBertModel(30522, 768, 512, 0.1)
token_embeddings = embedding_object(input_ids)
print(token_embeddings.shape)

torch.Size([2, 128, 768])


In [55]:
position_ids = torch.arange(input_ids.shape[-1], dtype=torch.long)

In [56]:
position_ids.shape

torch.Size([128])

In [57]:
position_ids.unsqueeze(0).expand(input_ids.shape[0], input_ids.shape[-1]).shape

torch.Size([2, 128])